# Fase 4 - Transformer destilado en CPU

Este notebook deja listo el flujo CPU-only para el transformer. Si no hay pesos locales del modelo, la precondicion queda explicita y el resto del entrenamiento no se ejecuta.

In [ ]:
from pathlib import Path
import sys

RAIZ_PROYECTO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.corpus_inmuebles import preparar_corpus_para_modelado
from src.evaluacion_modelos import construir_matriz_confusion_tabla, construir_reporte_clasificacion, construir_tabla_metricas
from src.infraestructura_cpu import configurar_torch_cpu, relevar_hardware, sugerir_tamanio_muestra
from src.transformer_cpu import (
    NOMBRE_MODELO_TRANSFORMER,
    cargar_modelo_transformer_para_clasificacion,
    cargar_tokenizador_transformer,
    construir_mapeo_etiquetas,
    crear_dataloaders_transformer,
    cuantizar_modelo_dinamicamente,
    entrenar_transformer_en_cpu,
    predecir_con_transformer,
    relevar_estado_modelo_local,
)


In [ ]:
estado_modelo = relevar_estado_modelo_local(NOMBRE_MODELO_TRANSFORMER)
estado_modelo


In [ ]:
hay_pesos_locales = bool(estado_modelo.loc[0, 'pesos_modelo_disponibles'])
print(f'Pesos locales disponibles: {hay_pesos_locales}')


In [ ]:
if hay_pesos_locales:
    RUTA_DATOS = RAIZ_PROYECTO / 'data' / 'entrenamiento.csv'
    resumen = relevar_hardware()
    configurar_torch_cpu()
    tamanio_muestra = min(6000, sugerir_tamanio_muestra(resumen.memoria_disponible_gb))

    df_muestra, df_entrenamiento, df_prueba = preparar_corpus_para_modelado(
        ruta_datos=RUTA_DATOS,
        tamanio_muestra=tamanio_muestra,
        tamanio_test=0.2,
        semilla=42,
    )

    etiqueta_a_id, id_a_etiqueta = construir_mapeo_etiquetas(df_muestra['property_type'])
    tokenizador = cargar_tokenizador_transformer()
    modelo = cargar_modelo_transformer_para_clasificacion(
        cantidad_etiquetas=len(etiqueta_a_id),
        etiqueta_a_id=etiqueta_a_id,
        id_a_etiqueta=id_a_etiqueta,
    )

    dl_train, dl_test = crear_dataloaders_transformer(
        df_entrenamiento,
        df_prueba,
        tokenizador,
        etiqueta_a_id=etiqueta_a_id,
        batch_size=4,
        longitud_maxima=128,
    )

    historial = entrenar_transformer_en_cpu(modelo, dl_train, epochs=1)
    display(historial)

    modelo_cuantizado = cuantizar_modelo_dinamicamente(modelo)
    pred_ids = predecir_con_transformer(modelo_cuantizado, dl_test)
    predicciones = [id_a_etiqueta[indice] for indice in pred_ids]

    print(construir_reporte_clasificacion(df_prueba['property_type'], predicciones))
    display(construir_tabla_metricas(df_prueba['property_type'], predicciones))
    display(
        construir_matriz_confusion_tabla(
            df_prueba['property_type'],
            predicciones,
            etiquetas_ordenadas=['Departamento', 'Casa', 'PH'],
        )
    )
else:
    print('No hay pesos locales del modelo. El notebook queda listo para entrenar cuando el modelo completo este cacheado o se habilite descarga.')
